# 📗 EDA 기초 — 결합·집계로 데이터 요약

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

> 📚 **pandas 공식 문서**: https://pandas.pydata.org/docs/ · 그룹 집계 [Group by: split-apply-combine](https://pandas.pydata.org/docs/user_guide/groupby.html) · 표 결합 [Merge, join, concatenate](https://pandas.pydata.org/docs/user_guide/merging.html) · API 검색 [API reference](https://pandas.pydata.org/docs/reference/index.html)

## 🎯 오늘의 목표
- [ ] `groupby` 로 데이터를 그룹으로 묶어 요약한다.
- [ ] 여러 키·여러 집계(`agg`)를 한 번에 계산한다.
- [ ] **`lambda`·사용자 함수**로 목록에 없는 집계를 직접 만든다.
- [ ] `pivot_table` 로 행·열 교차 요약표를 만든다.
- [ ] `crosstab` 으로 빈도표(교차표)를 만든다.
- [ ] `merge` 로 두 표를 공통 키로 이어 붙인다.
- [ ] `concat` 으로 표를 위·아래(또는 옆)로 이어 붙인다.
- [ ] 요약 결과에서 **인사이트**를 뽑아낸다.

## ⏪ 복습 — 지난 단원: pandas 기초
지난 단원에서는 표 한 장을 다루는 기본기를 익혔습니다.
- `pd.read_csv` 로 파일을 읽고, `head`·`info`·`describe` 로 훑어봤죠.
- 대괄호·`loc`·`iloc` 로 원하는 열·행을 골라내고, **불리언 필터**로 조건에 맞는 행만 추렸습니다.
- 결측치(`fillna`)와 이상치를 정제해 데이터를 깨끗하게 만들었습니다.

이제 깨끗한 표 한 장을 넘어, **"요일별 평균은?", "성별·흡연 여부로 나눠 보면?"** 같은 **그룹 단위 질문**에 답할 차례입니다. 그리고 흩어진 두 표를 **하나로 잇는 법**(merge·concat)도 배웁니다. 오늘은 레스토랑 팁 데이터로 **요약의 기술**을 익힙니다.

## 오늘의 데이터 — 레스토랑 팁 기록
`restaurant.csv` 는 어느 식당의 영수증 244건입니다. 손님이 낸 **팁(tip)** 이 요일·시간대·인원수에 따라 어떻게 달라지는지 살펴봅니다.

- `total_bill` — 전체 식사 금액(달러)
- `tip` — 팁 금액(달러)
- `sex` — 결제자 성별 (Male / Female)
- `smoker` — 흡연석 여부 (Yes / No)
- `day` — 요일 (Thur / Fri / Sat / Sun)
- `time` — 시간대 (Lunch / Dinner)
- `size` — 함께 온 인원수

In [ ]:
# [제공 코드] 오늘 내내 쓸 pandas 를 불러옵니다.
import pandas as pd

In [ ]:
# 데이터를 불러와 먼저 살펴봅니다 — 앞부분·구조·수치 요약
df = pd.read_csv("data/restaurant.csv")
print("행·열 크기:", df.shape)   # (244, 7) — 244행 7열
print("\n[앞 5행] head()")
display(df.head())
print("\n[열·자료형·결측] info()")
df.info()
print("\n[수치 요약] describe()")
display(df.describe())

---
# 1. groupby — 그룹으로 묶어 요약하기

## 왜 필요할까요?
"전체 팁 평균"은 `df['tip'].mean()` 한 줄이면 됩니다. 하지만 실무 질문은 대부분 **"~별로"** 입니다 — "**요일별** 평균 팁은?", "**인원수별** 평균 식사 금액은?".

이럴 때 `groupby` 를 씁니다. **① 같은 값끼리 그룹으로 묶고 → ② 각 그룹을 하나의 숫자로 요약**하는 2단계 도구예요.

| 문법 | 하는 일 |
|---|---|
| `df.groupby('day')['tip'].mean()` | 요일별 팁 **평균** |
| `df.groupby('day')['tip'].sum()` | 요일별 팁 **합계** |
| `df.groupby('day').size()` | 요일별 **행 개수**(그룹 크기) |
| `df.groupby('day')['tip'].count()` | 요일별 결측 아닌 값 개수 |

## 무엇을 그룹 기준으로 삼을까 — 기준 자리엔 범주형
`groupby` 의 **기준(키)** 자리에는 **범주형**이 옵니다 — 요일·성별·시간대처럼 **값이 몇 종류로 반복되는** 열이죠. 그리고 **집계할 값** 자리에는 보통 **수치형**이 옵니다.

| 자리 | 어떤 열이 오나 | 이 데이터에서 |
|---|---|---|
| **기준** — `groupby('...')` | **범주형** (값이 몇 종류로 **반복**되는 열) | `day` · `time` · `sex` · `smoker` |
| **집계 대상** — `['...']` | **주로 수치형** (평균·합계를 낼 수 있는 열) — 단 **범주형도 가능**(개수·종류 수·최빈값) | 수치형 `tip` · `total_bill` / 범주형 `sex` · `smoker` |

- **왜 기준이 범주형이어야 할까요?** 같은 값이 여러 번 나와야 **묶을 것**이 생기기 때문입니다. 요일은 244행에 4종류뿐이라 바구니 4개가 만들어집니다.
- 엄밀히 말하면 "**값이 반복되는 이산형**"이면 됩니다. 그래서 **인원수(`size`)처럼 정수로 딱 떨어지는 수치형도 기준이 될 수 있습니다**(1·2·3·4명처럼 종류가 적으니까요 — 바로 아래에서 해 봅니다).
- 반대로 **연속형**(`total_bill` 같은 금액)을 **그대로** 기준으로 쓰면 안 됩니다. 값이 거의 다 달라 **그룹이 행 수만큼 생기고** 요약이 되지 않습니다. 이럴 때는 **구간으로 묶어 범주로 바꾼 뒤**(`pd.cut`) 기준으로 씁니다.
- **집계 대상은 주로 수치형이지만, 범주형도 올 수 있습니다.** 평균·합계는 못 내도 **범주를 요약하는 방법이 따로 있으니까요** — 개수(`count`·`size`), 종류 수(`nunique`), 그리고 **최빈값(`mode`, 가장 흔한 값)**. "요일별로 가장 흔한 손님 유형은?" 같은 질문이 여기 해당합니다(세는 집계는 바로 아래에서, **최빈값은 3절**에서 해 봅니다).
- 단 범주형에 `mean` 을 시키면 **에러**가 납니다 — `TypeError: agg function failed [how->mean,dtype->object]`. 조용히 틀린 값이 나오는 게 아니라 확실히 알려 주니, "평균이 될 열인가"만 스스로 물으면 됩니다.

> `groupby('day')` 의 결과 인덱스는 **요일**이 됩니다. 244행이 요일 4줄로 줄어들죠 — 이게 "요약"입니다.

<img src="images/groupby_split_apply_combine.png" alt="groupby 3단계 — 나누고 계산하고 합치기" width="900"/>

In [ ]:
# 요일별 팁 평균 — groupby 로 244행을 요일 4줄로 요약
day_tip = df.groupby('day')['tip'].mean()
print(day_tip.round(3))

# 요약한 결과도 결국 표(Series)라 정렬할 수 있습니다 — 높은 순으로 줄 세우면 순위가 한눈에.
print('\n--- 평균 팁 높은 순 ---')
print(day_tip.sort_values(ascending=False).round(3))

In [ ]:
# 같은 방식으로 합계·그룹 크기도 구할 수 있습니다.
print("요일별 팁 합계:")
print(df.groupby('day')['tip'].sum())
print("\n요일별 주문 건수(size):")
print(df.groupby('day').size())

In [ ]:
# 인원수(size 열)별 평균 식사 금액 — 숫자 열로도 그룹을 만들 수 있어요.
# 주의: 'size' 는 인원수 '열 이름'입니다. groupby(...).size() 의 그룹 크기 메서드와 이름만 같아요.
# 인원이 많을수록 식사 금액이 커지는 경향이 보입니다.
df.groupby('size')['total_bill'].mean()

In [ ]:
# 연속형(금액)을 그대로 기준으로 쓰면? — 그룹이 행 수만큼 쪼개져 '요약'이 되지 않습니다.
print('금액 종류 수:', df['total_bill'].nunique(), '/ 전체 행 수:', len(df))

# 그래서 연속형은 '구간'으로 묶어 범주로 바꾼 뒤 기준으로 씁니다 (pd.cut).
# 원본 df 를 건드리지 않으려고 assign 으로 새 표를 만듭니다.
binned = df.assign(금액대=pd.cut(df['total_bill'], bins=[0, 15, 30, 45, 60],
                                labels=['15달러 이하', '15~30', '30~45', '45 초과']))
print()
print(binned.groupby('금액대', observed=True)['tip'].mean().round(2))
# 금액대가 올라갈수록 팁도 함께 오릅니다 — 구간으로 묶으니 경향이 드러납니다.

In [ ]:
# 집계 대상이 범주형일 때 — 평균 대신 '세는' 집계를 쓴다
print('요일별 흡연석 값의 종류 수(nunique)와 건수(count):')
display(df.groupby('day')['smoker'].agg(['nunique', 'count']))

print('요일별 흡연석 구성 — value_counts 로 값마다 몇 건인지:')
display(df.groupby('day')['smoker'].value_counts())

# 범주형에 평균을 시키면? — 조용히 틀리지 않고 에러로 알려 준다
try:
    df.groupby('day')['smoker'].mean()
except TypeError as e:
    print('범주형에 mean ->', type(e).__name__, ':', e)

# '가장 흔한 값(최빈값)'을 뽑는 방법은 3절(groupby 사용자 정의 집계)에서 다룹니다.

### 🖐️ 함께 따라하기 — 프로그램별 평균 운동시간
데모는 **레스토랑 팁** 데이터였죠. 따라하기는 **다른 도메인 — 피트니스 센터 이용 기록**(`gym_visits.csv`)으로 같은 기술을 연습합니다. 먼저 데이터를 불러와 훑어본 뒤 그룹 요약을 해 봅시다.

> ⚠️ 이 셀에서 만드는 `gym` 을 **이후 따라하기에서 계속 씁니다** — 건너뛰지 말고 꼭 실행하고 넘어가세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# ※ 데모는 '레스토랑 팁'이었죠. 이번엔 다른 데이터(피트니스 센터 이용 기록)로 연습합니다.
# 1) pd.read_csv 로 data/gym_visits.csv 를 읽어 변수 gym 에 담는다
# 2) head() 로 앞부분을, info() 로 열·자료형을 먼저 살펴본다
# 3) gym 을 '프로그램' 으로 그룹화해 '운동시간' 의 평균을 구한다
# 4) 결과를 출력해 평균 운동시간이 가장 긴 프로그램을 확인한다

### ✅ 바로 확인 퀴즈
**1.** `df.groupby('day')['tip'].mean()` 의 결과는 왜 요일마다 **한 줄**(총 4줄)일까요?

<details><summary>정답 보기</summary>

groupby 가 같은 요일끼리 그룹으로 묶어 **각 그룹을 평균값 하나로 요약**하기 때문입니다. 244행이 요일 4그룹으로 줄어듭니다.

</details>

**2.** 위 데이터에서 평균 팁이 **가장 높은 요일**은 어디일까요?

<details><summary>정답 보기</summary>

**Sun(일요일)**, 약 3.255달러입니다. (Fri 2.735 < Thur 2.771 < Sat 2.993 < Sun 3.255)

</details>

**3.** 요일별 **주문 건수**를 구하려면 어떤 메서드를 쓸까요?

<details><summary>정답 보기</summary>

`df.groupby('day').size()` 입니다. size 는 결측 포함 그룹 전체 행 수, 특정 열의 `.count()` 는 그 열의 결측을 뺀 개수예요.

</details>

---
# 2. groupby 심화 — 여러 키·여러 집계

## 왜 필요할까요?
그룹 기준이 하나로 부족할 때가 있습니다 — "**요일 × 시간대**별 평균은?". 그리고 한 그룹을 여러 각도로 보고 싶을 때도 있죠 — "평균**과** 합계**와** 개수를 한 번에".

- **여러 키로 묶기**: `groupby(['day', 'time'])` — 리스트로 여러 열을 넘기면 조합별로 그룹이 만들어집니다.
- **여러 집계 한 번에**: `.agg(['mean', 'sum', 'count'])` — 집계 함수 이름을 리스트로 넘깁니다.
- **그룹 키를 열로 두기**: `as_index=False` — 그룹 키를 인덱스가 아니라 **일반 열**로 돌려받아 이후 다루기 편하게 합니다.

| 문법 | 하는 일 |
|---|---|
| `df.groupby(['day','time'])['total_bill'].mean()` | 요일×시간대별 평균 |
| `df.groupby('day')['tip'].agg(['mean','sum','count'])` | 한 그룹에 여러 집계 동시에 |
| `df.groupby('day', as_index=False)['tip'].mean()` | 그룹 키(day)를 일반 열로 |

In [ ]:
# 여러 키 — 요일 × 시간대 조합별 평균 식사 금액
df.groupby(['day', 'time'])['total_bill'].mean()

In [ ]:
# 여러 집계 — 요일별 팁의 평균·합계·개수를 한 표로 (리스트 형태)
display(df.groupby('day')['tip'].agg(['mean', 'sum', 'count']).round(3))

# 열마다 다른 집계가 필요할 땐 딕셔너리로 — {'열이름': '집계함수'}
# 실무에서 가장 많이 쓰는 형태입니다: 금액은 평균, 팁은 합계, 인원은 최댓값처럼.
display(df.groupby('day').agg({
    'total_bill': 'mean',    # 식사 금액은 평균
    'tip': 'sum',            # 팁은 합계
    'size': 'max',           # 인원수는 최댓값
}).round(3))

In [ ]:
# as_index=False — 그룹 키(day)가 인덱스가 아니라 일반 열로 나옵니다.
df.groupby('day', as_index=False)['tip'].mean()

### 🖐️ 함께 따라하기 — 프로그램 × 회원등급
키를 **두 개**로 묶어 봅시다. 프로그램별로, 또 그 안에서 회원등급별로 나눠 평균 소모칼로리를 구합니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) gym 을 '프로그램'·'회원등급' 두 열로 그룹화한다 (리스트로 묶어 전달)
# 2) 그 그룹의 '소모칼로리' 평균을 구한다
# 3) 결과를 출력한다 (프로그램 4종 × 등급 2종 = 8줄이 나오는지 확인)

### ✅ 바로 확인 퀴즈
**1.** `df.groupby(['day','time'])` 처럼 키를 **리스트**로 주면 결과는 어떻게 될까요?

<details><summary>정답 보기</summary>

요일과 시간대의 **조합**마다 그룹이 생깁니다. 결과 인덱스가 (day, time) 2단계가 되죠.

</details>

**2.** 요일별 팁의 **평균·합계·개수**를 한 번에 구하는 코드는?

<details><summary>정답 보기</summary>

`df.groupby('day')['tip'].agg(['mean','sum','count'])` — 집계 함수 이름을 리스트로 넘깁니다.

</details>

---
# 3. groupby 사용자 정의 집계 — lambda 와 내 함수로

## 왜 필요할까요?
지금까지 쓴 `'mean'`·`'sum'`·`'max'` 는 **pandas 가 미리 만들어 둔 집계**입니다. 그런데 실무 질문은 그 목록에 없는 것이 많습니다.

- "요일별로 팁이 **가장 많은 손님과 가장 적은 손님의 차이**는?" → 이름이 없는 계산(`최댓값 - 최솟값`)
- "팁을 **3달러 이상 낸 손님의 비율**은?" → 조건을 세는 계산
- "**인원수를 감안한** 팁 평균은?" → 두 열을 함께 봐야 하는 계산

이럴 때는 **계산 방법을 내가 직접 써서 건네줍니다.** 3일차에 배운 **lambda**(한 줄 함수)와 6일차에 배운 **사용자 정의 함수(`def`)** 가 여기서 쓰입니다.

### 문법
| 문법 | 하는 일 |
|---|---|
| `.agg(lambda s: s.max() - s.min())` | 그룹의 **그 열**(Series)을 받아 숫자 하나로 |
| `.agg(['mean', lambda s: ...])` | 이름표와 내 계산을 **섞어** 쓰기 |
| `.agg(새이름=('열', 함수))` | 결과 **열 이름을 직접** 지정 |
| `.agg(내함수)` | `def` 로 만든 함수 — **함수 이름이 열 이름**이 된다 |
| `.apply(내함수)` | 그룹의 **표 전체**(여러 열)를 받아 아무 모양으로든 |

> **`lambda s:` 의 `s` 에는 무엇이 담길까요?** 그룹 하나의 **그 열 값들**입니다. 요일이 4종류면 함수가 **4번** 호출되고, 그때마다 `s` 에는 그 요일의 팁만 담겨 있습니다. `s` 는 Series 라 `s.max()`·`s.mean()`·`(s >= 3).mean()` 을 그대로 쓸 수 있어요.

In [ ]:
# ① lambda — 이름이 없는 계산: 그룹 안의 최댓값 - 최솟값(범위)
tip_range = df.groupby('day')['tip'].agg(lambda s: s.max() - s.min())
print(tip_range.round(2))
# 토요일이 9.00 으로 가장 넓습니다 — 아주 많이 준 손님과 아주 적게 준 손님이 함께 있다는 뜻이죠.
# 평균(2.99)만 봤을 때는 보이지 않던 사실입니다.

In [ ]:
# ② 이름표와 내 계산을 한 표에 + 결과 열 이름 직접 붙이기
#    형태:  새열이름=('집계할 열', 집계방법)
summary = df.groupby('day').agg(
    평균팁=('tip', 'mean'),                       # 이름표(미리 만들어진 집계)
    팁범위=('tip', lambda s: s.max() - s.min()),  # 내가 쓴 계산
    건수=('tip', 'size'),
).round(2)
display(summary)
# 이름을 직접 주지 않으면 lambda 로 만든 열엔 '<lambda_0>' 같은 이름이 붙어 알아보기 어렵습니다.

In [ ]:
# ③ def 로 만든 함수 — 이름이 붙고, 여러 곳에서 다시 쓸 수 있다
def generous_tip_ratio(s):
    """팁을 3달러 이상 낸 손님의 비율(%)."""
    return (s >= 3).mean() * 100      # True/False 의 평균 = True 의 비율

print(df.groupby('day')['tip'].agg(generous_tip_ratio).round(1))
# 열 이름이 함수 이름 그대로 붙습니다 — 그래서 함수 이름을 알아보기 쉽게 지어야 합니다.
# 일요일이 61.8% 로 가장 후하네요. (열 이름을 한글로 두고 싶으면 위 ②의 named aggregation 을 쓰세요.)

### `agg` 와 `apply` — 무엇을 받아 무엇을 돌려주나

| | 받는 것 | 돌려주는 것 | 언제 쓰나 |
|---|---|---|---|
| `agg` | 그룹의 **한 열**(Series) | **숫자 하나** | 열 하나로 계산이 끝날 때 |
| `apply` | 그룹의 **표 전체**(DataFrame) | **아무 모양**(숫자 하나·여러 행) | 여러 열을 함께 봐야 할 때 |

> 헷갈리면 이렇게 기억하세요 — **한 열이면 `agg`, 여러 열이면 `apply`.**

### 잠깐 — 가중 평균이 뭔가요?

**가중 평균**은 값마다 **무게(가중치)를 달리 주어** 내는 평균입니다. 우리가 늘 쓰던 단순 평균은 **모든 값에 똑같은 무게 1**을 준 특수한 경우예요.

```text
가중 평균 = (값1×무게1 + 값2×무게2 + …) ÷ (무게1 + 무게2 + …)
```

**왜 필요할까요?** "**무엇을 한 표로 셀 것인가**"가 달라지기 때문입니다. 테이블 두 개가 있다고 해 봅시다 — **2명이 팁 3달러**, **8명이 팁 1달러**를 냈습니다.

| 무엇을 한 표로? | 계산 | 결과 |
|---|---|---|
| **테이블**을 한 표로 | (3 + 1) ÷ 2 | **2.0달러** |
| **사람**을 한 표로 | (3×2 + 1×8) ÷ (2 + 8) | **1.4달러** |

같은 데이터인데 답이 다릅니다. **둘 다 맞는 계산**이고, 무엇을 알고 싶은지가 어느 쪽을 쓸지 정합니다.

우리 데이터의 `tip` 은 **테이블 단위**로 기록돼 있습니다. 그래서 그냥 평균을 내면 "테이블당 평균 팁"이고, 인원수(`size`)를 가중치로 주면 "**사람 수를 감안한** 평균 팁"이 됩니다. 아래에서 먼저 손으로 계산해 본 뒤, 바로 이어서 요일별로 두 값을 나란히 비교합니다.

> 가중 평균은 8일차 기술통계에서 대푯값을 다룰 때 다시 만납니다 — 여기서 감을 잡아 두세요.

In [ ]:
# 가중 평균을 손으로 계산해 보기 — 위 표의 그 예시입니다.
tips_paid = [3, 1]     # 각 테이블이 낸 팁
people = [2, 8]        # 그 테이블의 인원수 = 가중치

simple = sum(tips_paid) / len(tips_paid)
weighted = sum(t * n for t, n in zip(tips_paid, people)) / sum(people)
print(f'단순 평균: {simple}달러   <- 테이블 하나를 한 표로 셈')
print(f'가중 평균: {weighted}달러   <- 사람 한 명을 한 표로 셈')

# 이 계산을 numpy 가 한 줄로 해 줍니다 (아래 ④ 에서 이 함수를 씁니다).
import numpy as np
print('\nnp.average(값, weights=무게):', np.average(tips_paid, weights=people))
print('무게를 안 주면 단순 평균과 같아집니다:', np.average(tips_paid))

In [ ]:
# ④ agg 로는 안 되는 일 — 여러 열을 함께 보기
import numpy as np

def weighted_tip_mean(g):                # g 는 그룹 하나의 DataFrame (여러 열이 다 들어 있다)
    return np.average(g['tip'], weights=g['size'])    # 인원수를 가중치로

weighted = df.groupby('day')[['tip', 'size']].apply(weighted_tip_mean)
compare = pd.DataFrame({'단순평균': df.groupby('day')['tip'].mean(),
                        '인원가중평균': weighted})
display(compare.round(3))
# 목요일이 2.771 -> 3.114 로 크게 오릅니다 — 인원이 많은 테이블이 팁을 더 냈다는 뜻이죠.

In [ ]:
# apply 는 '숫자 하나'가 아니어도 됩니다 — 그룹마다 여러 행을 돌려줄 수도 있어요(agg 로는 불가능).
top2 = df.groupby('day')[['total_bill', 'tip']].apply(lambda g: g.nlargest(2, 'tip'))
display(top2)
# 요일마다 팁 상위 2건이 뽑혔습니다 — 4요일 x 2건 = 8행.

In [ ]:
# ⑤ 집계 대상이 '범주형'일 때 — 최빈값(가장 흔한 값)
# mode() 는 동률이면 여러 개를 돌려주므로 [0] 으로 첫 값을 집는다.
print('요일별 가장 흔한 흡연석 여부:')
print(df.groupby('day')['smoker'].agg(lambda s: s.mode()[0]))

print('\n요일별 가장 흔한 결제자 성별:')
print(df.groupby('day')['sex'].agg(lambda s: s.mode()[0]))
# 목요일만 Female 이 최빈값입니다 — 목요일은 점심 장사가 대부분이라는 사실과 이어 볼 만하죠.

In [ ]:
# 범주형을 요약하는 방법은 최빈값 말고도 여러 가지입니다.
# (1) 그 그룹에 어떤 값들이 있었나 — 문자열로 이어 붙이기
print('요일별 등장한 시간대:')
print(df.groupby('day')['time'].agg(lambda s: ', '.join(sorted(s.unique()))))

# (2) 최빈값이 '얼마나' 우세한가 — 점유율(%)까지 함께 봐야 안전하다
share = df.groupby('day')['smoker'].agg(lambda s: s.value_counts(normalize=True).iloc[0] * 100)
print('\n최빈값의 점유율(%):')
print(share.round(1))
# 토요일은 51.7% — 최빈값이 No 이지만 Yes 와 거의 반반입니다.
# 최빈값만 보고 '토요일은 비흡연석 손님이다'라고 말하면 위험하다는 뜻이죠.

# (3) 두 범주를 합쳐 '조합'의 최빈값 — 가장 흔한 손님 유형은?
customer = df.assign(유형=df['sex'] + '·' + df['smoker'])
display(customer.groupby('day')['유형'].agg(
    가장흔한유형=lambda s: s.mode()[0],
    점유율=lambda s: round(s.value_counts(normalize=True).iloc[0] * 100, 1)))
# 목요일은 'Female·No'(40.3%), 일요일은 'Male·No'(56.6%) 가 가장 흔합니다.
# 요일마다 오는 손님의 '색깔'이 다르다는 뜻이죠 — 평균 팁만 봐서는 안 보이던 사실입니다.

In [ ]:
# ⑥ 수치형과 범주형을 '한 표에' 섞기 — ②의 이름 붙이기를 그대로 씁니다
mixed = df.groupby('day').agg(
    평균팁=('tip', 'mean'),                        # 수치형 -> 평균
    최빈_흡연석=('smoker', lambda s: s.mode()[0]),  # 범주형 -> 최빈값
    시간대_종류=('time', 'nunique'),                # 범주형 -> 종류 수
    건수=('tip', 'size'),
).round(2)
display(mixed)
# 한 표에서 '금요일은 팁이 가장 낮고(2.73), 흡연석이 최빈값이며, 점심·저녁이 다 있다'까지 읽힙니다.

### 🖐️ 함께 따라하기 — 프로그램별 소모칼로리 요약표
데모는 레스토랑 팁이었죠. 이번엔 **피트니스 데이터**로, 이름표 집계·내가 쓴 계산·**범주형 집계**를 **한 표에** 담아 봅시다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) gym 을 '프로그램' 으로 그룹화한다
# 2) agg 에 이름을 직접 붙여 세 열을 만든다
#    - 평균칼로리 : '소모칼로리' 의 평균
#    - 칼로리범위 : '소모칼로리' 의 최댓값 - 최솟값   (lambda 로 작성)
#    - 방문건수   : '소모칼로리' 의 개수(size)
# 3) 소수 1자리로 반올림해 출력한다 (프로그램 4종 = 4줄)
# 4) '평균은 가장 높지 않은데 범위는 가장 넓은' 프로그램을 찾아본다
# 5) 같은 표에 범주형 집계 두 열을 더 붙인다
#    - 주요등급   : '회원등급' 의 최빈값        (mode()[0])
#    - 등급점유율 : 그 최빈값이 차지하는 비율(%)  (value_counts(normalize=True).iloc[0] * 100)
# 6) 네 프로그램의 주요등급이 모두 같은지, 점유율은 어떻게 다른지 살펴본다

### ✅ 바로 확인 퀴즈
**1.** `df.groupby('day')['tip'].agg(lambda s: ...)` 에서 `s` 에는 무엇이 담기나요?

<details><summary>정답 보기</summary>

**그룹 하나의 `tip` 값들**(Series)입니다. 요일이 4종류면 함수가 4번 불리고, 그때마다 그 요일의 팁만 `s` 에 들어옵니다.

</details>

**2.** lambda 로 만든 결과 열에 `<lambda_0>` 대신 **원하는 이름**을 붙이려면?

<details><summary>정답 보기</summary>

`.agg(새이름=('열이름', 함수))` 형태로 씁니다. 예: `.agg(팁범위=('tip', lambda s: s.max() - s.min()))`

</details>

**3.** "인원수를 가중치로 한 팁 평균"처럼 **두 열을 함께** 써야 하는 계산은 `agg` 와 `apply` 중 무엇으로 하나요?

<details><summary>정답 보기</summary>

**`apply`** 입니다. `agg` 는 **한 열**(Series)만 받지만, `apply` 는 그룹의 **표 전체**(DataFrame)를 받아 여러 열을 함께 쓸 수 있습니다.

</details>

---
# 4. pivot_table — 행·열 교차 요약표

## 왜 필요할까요? — groupby 와 무엇이 다른가
`groupby(['day','time'])` 와 `pivot_table` 은 **똑같은 계산**을 합니다. 다른 것은 **결과를 어디에 놓느냐** 하나뿐이에요.

- `groupby(['day','time'])` → **조합 하나가 한 줄**입니다. 요일×시간대 조합이 6개면 6줄이 세로로 쌓이고, 인덱스가 (요일, 시간대) **2단**이 됩니다.
- `pivot_table(index='day', columns='time')` → **첫 기준은 행으로, 둘째 기준은 열로** 갑니다. 요일 4행 × 시간대 2열짜리 **격자**가 되죠.

| | `groupby` (키 두 개) | `pivot_table` |
|---|---|---|
| 조합이 놓이는 곳 | 전부 **행** — 아래로 쌓임 | **행 × 열** 격자 |
| 결과 모양 | 6줄짜리 Series (인덱스 2단) | 4행 × 2열 DataFrame |
| **없는 조합** | **줄 자체가 없음** — 주말 점심은 아예 안 보임 | **빈 칸(NaN)** 으로 드러남 |
| 잘 맞는 쓰임 | 이어서 **계산·필터·정렬·병합** 할 때 | 사람이 **눈으로 비교**할 때, 히트맵 입력 |

> **정말 같은 계산일까요?** `groupby` 결과에 `.unstack()` 을 붙이면 **pivot_table 과 글자 하나까지 같은 표**가 나옵니다(바로 아래에서 확인합니다). 그러니 둘 중 무엇을 쓸지는 **맞고 틀리고의 문제가 아니라 "이 표를 어디에 쓸 것인가"의 문제**입니다.

> 그리고 **없는 조합을 다루는 방식이 다른 것**은 실제로 쓸모가 있습니다. "주말 점심 장사를 안 한다"는 사실은 격자의 **빈 칸**으로는 눈에 띄지만, groupby 결과에서는 줄이 없어 **눈치채기 어렵습니다.**

<img src="images/groupby_vs_pivot.png" alt="groupby 두 키 결과와 pivot_table 결과 비교 — 같은 숫자, 다른 배치" width="960"/>

| 인자 | 뜻 |
|---|---|
| `index='day'` | 표의 **행**이 될 기준 |
| `columns='time'` | 표의 **열**이 될 기준 |
| `values='total_bill'` | 셀에 채울 **값** (수치형이 일반적, 범주형도 가능) |
| `aggfunc='mean'` | 값을 요약할 방법(평균·합계 등) — **`lambda`·내 함수도 가능** |
| `margins=True` | 행·열 **총계(All)** 를 덧붙임 |

- **`index`·`columns` 자리에는 범주형**(요일·시간대)이 옵니다 — groupby 의 기준과 같은 이유입니다. 값이 반복돼야 격자의 칸이 생기니까요.
- **`aggfunc` 에도 3절의 사용자 정의 집계가 그대로 들어갑니다** — `lambda`·`def` 함수, 여러 집계 리스트, 열마다 다른 집계 딕셔너리까지 됩니다. 단 **groupby 의 이름 붙이기(`agg(새이름=('열', 함수))`)는 pivot_table 에 없습니다** — `TypeError` 가 나므로, 열 이름은 뒤에서 `.rename(columns=...)` 으로 바꿉니다.
- **`values` 자리도 주로 수치형이지만, 범주형이 올 수 있습니다** — groupby 와 똑같은 원리입니다. `aggfunc='count'`·`'nunique'` 로 세거나, `aggfunc=lambda s: s.mode()[0]` 로 최빈값을 채우면 **칸에 글자가 들어간 격자표**가 나옵니다(아래에서 해 봅니다). 물론 범주형에 `'mean'` 을 주면 여기서도 에러입니다.
- 연속형을 행·열에 두면 칸이 행 수만큼 생겨 표가 무의미해집니다 — `pd.cut` 으로 구간을 만들어 쓰세요(1번 참고).

<img src="images/pivot_long_to_wide.png" alt="pivot_table — 긴 표를 행×열 격자로" width="900"/>

In [ ]:
# groupby 와 pivot_table 은 같은 계산 — 결과가 놓이는 자리만 다릅니다.
by_group = df.groupby(['day', 'time'])['total_bill'].mean()
print('[groupby] 조합 하나가 한 줄 —', len(by_group), '줄 / 인덱스', by_group.index.nlevels, '단')
print(by_group.round(2))

by_pivot = df.pivot_table(index='day', columns='time', values='total_bill')
print('\n[pivot_table] 행 x 열 격자 —', by_pivot.shape[0], '행', by_pivot.shape[1], '열')
display(by_pivot.round(2))

# 정말 같은 계산인지 확인 — groupby 결과를 격자로 펼쳐(unstack) 견줍니다.
print('두 결과가 완전히 같은가:', by_group.unstack().equals(by_pivot))
# groupby 는 6줄인데 격자는 4x2=8칸입니다. 차이인 2칸이 '주말 점심' — 주문이 없어
# groupby 에는 줄이 아예 없고, 격자에는 NaN 빈 칸으로 남습니다.

In [ ]:
# 요일 × 시간대 평균 식사 금액을 격자표로
df.pivot_table(index='day', columns='time', values='total_bill', aggfunc='mean')

In [ ]:
# 결과에 NaN(빈 칸)이 보이나요?
# Sat·Sun 의 Lunch 칸이 NaN 입니다 — 이 데이터에 '주말 점심' 주문이 한 건도 없어서예요.
# NaN 은 "값이 틀렸다"가 아니라 "해당 조합이 아예 없다"는 뜻입니다.
# margins=True 로 행·열 총계(All)도 함께 봅시다.
display(df.pivot_table(index='day', columns='time', values='total_bill',
                       aggfunc='mean', margins=True).round(2))

# fill_value= 로 빈 칸을 원하는 값으로 채울 수 있습니다 (여기선 0).
# ⚠️ 단, '주문이 없다'와 '금액이 0원이다'는 다른 뜻이니 0으로 채울지는 신중히 판단하세요.
#    개수(count)를 셀 때는 0이 자연스럽지만, 평균 금액을 0으로 채우면 해석이 왜곡됩니다.
display(df.pivot_table(index='day', columns='time', values='total_bill',
                       aggfunc='count', fill_value=0))

In [ ]:
# aggfunc 에도 3절의 사용자 정의 집계가 그대로 들어갑니다.
def tip_range(s):
    """그룹 안 팁의 최댓값 - 최솟값."""
    return s.max() - s.min()

print('[요일 x 시간대 팁 범위 — lambda]')
display(df.pivot_table(index='day', columns='time', values='tip',
                       aggfunc=lambda s: s.max() - s.min()).round(2))
# 목요일 저녁이 0.00 인 이유는 주문이 1건뿐이라 최댓값과 최솟값이 같기 때문입니다.

print('[여러 집계를 한 번에 — def 함수는 함수 이름이 열 라벨이 된다]')
display(df.pivot_table(index='day', values='tip', aggfunc=['mean', tip_range]).round(2))
# lambda 를 리스트에 넣으면 열 이름이 '<lambda>' 가 됩니다 — 재사용할 계산은 def 로 이름을 붙이는 게 좋아요.

print('[열마다 다른 집계 — 딕셔너리]')
display(df.pivot_table(index='day', values=['tip', 'total_bill'],
                       aggfunc={'tip': tip_range, 'total_bill': 'mean'}).round(2))

# 단, groupby 의 '이름 붙이기'는 pivot_table 에 없습니다 — 에러로 알려 주죠.
try:
    df.pivot_table(index='day', 팁범위=('tip', tip_range))
except TypeError as e:
    print('pivot_table 에 이름 붙이기 ->', type(e).__name__, ':', e)

# 그래서 이름은 뒤에서 바꿉니다.
display(df.pivot_table(index='day', values='tip', aggfunc=tip_range)
          .rename(columns={'tip': '팁범위'}).round(2))

In [ ]:
# values 에 범주형을 넣어도 됩니다 — 세거나(count·nunique), 최빈값을 채우거나
print('[흡연석 여부 건수 — aggfunc="count"]')
display(df.pivot_table(index='day', columns='time', values='smoker', aggfunc='count'))

print('[값의 종류 수 — aggfunc="nunique"]')
display(df.pivot_table(index='day', columns='time', values='smoker', aggfunc='nunique'))
# 목요일 저녁은 1 입니다 — 주문이 1건뿐이라 흡연석 값도 한 종류만 나왔죠.

print('[가장 흔한 흡연석 여부 — 칸에 글자가 들어간다]')
display(df.pivot_table(index='day', columns='time', values='smoker',
                       aggfunc=lambda s: s.mode()[0] if len(s) else None))
# 금요일은 점심·저녁 모두 Yes(흡연석)가 최빈값이고 나머지 요일은 No 입니다.
# 주말 점심 칸이 NaN 인 이유는 앞에서 본 것과 같아요 — 그 조합의 주문이 아예 없기 때문입니다.

# 한 번에 여러 집계도 됩니다 — aggfunc 에 리스트를 주면 열이 층으로 쌓입니다.
display(df.pivot_table(index='day', values='smoker', aggfunc=['count', 'nunique']))

### 🖐️ 함께 따라하기 — 요일 × 시간대 격자표
`pivot_table` 로 **요일(행) × 시간대(열)** 격자표를 만들어 봅시다. 평균으로 한 번, 그다음 **내가 쓴 계산(범위)** 으로 한 번 — `aggfunc` 만 바꾸면 됩니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) gym 의 pivot_table 을 쓴다 — 행(index)은 '요일', 열(columns)은 '시간대'
# 2) 값(values)은 '운동시간', 집계(aggfunc)는 평균으로 지정한다
# 3) 결과를 소수 1자리로 반올림해 출력한다 (7행 × 3열 격자가 나오는지 확인)
# 4) 같은 격자표를 aggfunc 만 바꿔 '운동시간 범위(최댓값 - 최솟값)'로 다시 만든다
#    - lambda 로 작성한다 (3절에서 쓴 것과 같은 형태)
# 5) 평균 격자표와 범위 격자표를 비교해 본다 — 평균은 비슷한데 범위가 큰 칸이 있는지
# 6) 평균 격자표에 margins=True 를 주어 총계 행·열(All)을 붙여 본다
#    -> 오른쪽 아래 All 칸은 '전체 평균'입니다. 리포트에 표를 넣을 때 거의 항상 함께 봅니다.

### ✅ 바로 확인 퀴즈
**1.** pivot_table 결과에서 Sat·Sun 의 Lunch 칸이 NaN 인 이유는?

<details><summary>정답 보기</summary>

이 데이터에 **주말 점심 주문이 없기 때문**입니다. 그 조합에 해당하는 행이 0건이라 집계할 값이 없어 NaN 이 됩니다.

</details>

**2.** 행·열의 **총계(All)** 까지 표에 넣으려면 어떤 인자를 줄까요?

<details><summary>정답 보기</summary>

`margins=True` 를 줍니다.

</details>

---
# 5. crosstab — 빈도표(교차표)

## 왜 필요할까요?
"요일별로 Lunch·Dinner 주문이 **각각 몇 건**인가?" 처럼 **개수를 세는** 교차표가 자주 필요합니다. `pd.crosstab` 은 이 **빈도표**에 특화된 도구예요.

- `pd.crosstab(df['day'], df['time'])` — 요일(행) × 시간대(열)별 **건수**.
- `normalize='index'` — 각 행을 **비율**(행 합이 1)로. 행 안에서의 구성비를 볼 때 유용.
- `crosstab` vs `pivot_table`: 둘 다 교차표지만, **crosstab 은 빈도(개수) 세기에 특화**되어 값(values) 인자가 필요 없습니다. 값을 집계(평균 등)하려면 pivot_table 을 씁니다.
- **행·열 자리 둘 다 범주형**입니다. crosstab 은 "그 조합이 **몇 번** 나왔나"를 세는 도구라, 양쪽 모두 값이 반복되는 열이어야 셀 것이 생깁니다. 그래서 pivot_table 과 달리 **집계할 값 열이 없어도** 됩니다 — 세는 것 자체가 값이니까요.

In [ ]:
# 요일 × 시간대 주문 건수 교차표
pd.crosstab(df['day'], df['time'])

In [ ]:
# normalize='index' — 각 요일 안에서 Lunch/Dinner 비율(행 합이 1)
# Thur 는 대부분 Lunch, Sat·Sun 은 전부 Dinner 임이 한눈에 보입니다.
display(pd.crosstab(df['day'], df['time'], normalize='index').round(3))

# normalize='columns' — 방향을 바꿔서, 각 시간대 안에서 요일 비율(열 합이 1)
# "점심 손님은 주로 무슨 요일에 오나?"처럼 질문이 달라지면 정규화 방향도 달라집니다.
display(pd.crosstab(df['day'], df['time'], normalize='columns').round(3))

### 🖐️ 함께 따라하기 — 프로그램 × 회원등급 빈도표
평균이 아니라 **몇 건인지**가 궁금할 때는 `crosstab` 입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) pd.crosstab 으로 행에 gym['프로그램'], 열에 gym['회원등급'] 을 놓아 방문 건수 교차표를 만든다
# 2) 결과를 출력한다
# 3) 이어서 normalize='index' 를 주면 각 프로그램 안에서의 등급 '비율'(행 합=1)이 나온다 — 그것도 출력해 본다

### ✅ 바로 확인 퀴즈
**1.** `pd.crosstab` 과 `pivot_table` 의 가장 큰 차이는?

<details><summary>정답 보기</summary>

crosstab 은 **빈도(개수) 세기**에 특화되어 값 인자 없이 조합별 건수를 셉니다. 값을 집계(평균 등)하려면 pivot_table 을 씁니다.

</details>

**2.** 각 행을 비율(행 합=1)로 바꾸려면 어떤 인자를 줄까요?

<details><summary>정답 보기</summary>

`normalize='index'` 인자를 줍니다. (열 기준은 `normalize='columns'`)

</details>

---
# 6. merge — 두 표를 키로 이어 붙이기

## 왜 필요할까요?
분석에 필요한 정보가 **여러 표에 흩어져** 있을 때가 많습니다. 예를 들어 팁 데이터엔 요일만 있고, "그 요일이 주말인가?"는 **다른 표**(`day_info.csv`)에 있죠. 공통 열(**키**, 여기선 `day`)을 기준으로 두 표를 옆으로 합치는 게 `merge` 입니다.

`how=` 로 **어느 쪽 키를 남길지** 정합니다.
- `how='inner'` (기본): **양쪽 모두에 있는** 키만 남깁니다(교집합).
- `how='left'`: **왼쪽 표의 키는 전부** 남기고, 오른쪽에 짝이 없으면 NaN.
- `how='right'`: **오른쪽 표의 키**를 전부 남깁니다.
- `how='outer'`: **양쪽의 모든 키**를 남깁니다(합집합).

<img src="images/merge_how_4.png" alt="merge 결합 방식 4종" width="820"/>

In [ ]:
# 병합할 작은 표를 읽어옵니다. 요일마다 '주말인지(is_weekend)' 정보가 있어요.
day_info = pd.read_csv("data/day_info.csv")
day_info

In [ ]:
# 'day' 를 키로 두 표를 병합 — 팁 데이터에 is_weekend·sort_order 열이 붙습니다.
merged = df.merge(day_info, on='day')
print("병합 결과 크기:", merged.shape)   # (244, 9) — 244행 유지, 열이 7->9로 늘어남
merged.head(3)

### `how=` — 짝이 없는 행을 어떻게 할까

위 예시는 `day_info` 에 4개 요일이 **모두** 있어서 행 수가 그대로였습니다. 하지만 실무의 참조표는 **비어 있는 키**가 흔합니다. 그때 `how=` 가 결과를 가릅니다.

| `how` | 남기는 행 | 짝이 없으면 |
|---|---|---|
| `'inner'`(기본) | 양쪽에 **다 있는** 키만 | 그 행이 **사라짐** |
| `'left'` | **왼쪽 표 전부** | 오른쪽 열이 `NaN` |
| `'right'` | **오른쪽 표 전부** | 왼쪽 열이 `NaN` |
| `'outer'` | **양쪽 전부** | 없는 쪽이 `NaN` |

In [ ]:
# how= 차이를 눈으로 — 일부러 '금요일 정보가 빠진' 참조표를 만들어 비교합니다.
day_info_partial = day_info[day_info['day'] != 'Fri']   # Fri 행이 없는 표
print("참조표에 있는 요일:", list(day_info_partial['day']))

inner_df = df.merge(day_info_partial, on='day', how='inner')   # 짝 있는 것만
left_df  = df.merge(day_info_partial, on='day', how='left')    # 왼쪽(주문)은 전부 유지

print("inner:", inner_df.shape, "-> 금요일 주문이 통째로 빠졌습니다")
print("left :", left_df.shape,  "-> 행 수는 유지, 대신 금요일은 정보가 NaN")
print("left 의 is_weekend 결측 개수:", left_df['is_weekend'].isna().sum(), "(= 금요일 주문 수)")

print("\n오른쪽 표 키가 중복 없이 하나씩인가:", day_info['day'].is_unique)

### 키 이름이 다르거나, 열 이름이 겹칠 때

실무에서 두 표를 붙일 때 걸리는 두 가지가 더 있습니다.

- **키 이름이 서로 다를 때** — 왼쪽은 `day`, 오른쪽은 `요일` 처럼 같은 뜻인데 이름이 다르면 `on=` 을 못 씁니다. 이때는 `left_on='day', right_on='요일'` 로 양쪽 키를 따로 지정합니다.
- **키 말고 다른 열 이름이 겹칠 때** — 두 표에 똑같이 `note` 열이 있으면 pandas 가 자동으로 `note_x`(왼쪽)·`note_y`(오른쪽)로 이름을 바꿉니다. 이 꼬리표를 `suffixes=('_주문', '_요일정보')` 처럼 **알아보기 쉽게** 지정할 수 있습니다.

In [ ]:
# 키 이름이 다른 경우 — left_on / right_on
day_info_kr = day_info.rename(columns={'day': '요일'})     # 오른쪽 표의 키 이름을 '요일'로 바꿔 둠
print("왼쪽 키: 'day' / 오른쪽 키: '요일'")
merged_kr = df.merge(day_info_kr, left_on='day', right_on='요일')
print("병합 성공:", merged_kr.shape, "— 키가 양쪽 다 남습니다(day·요일)")

# 열 이름이 겹치는 경우 — suffixes
left = df[['day', 'total_bill']].head(3).copy()
right = day_info.copy()
left['note'] = '주문기록'        # 양쪽에 똑같은 이름의 열을 일부러 만들어 봅니다
right['note'] = '요일정보'

print("\n--- suffixes 를 안 주면 자동으로 _x / _y ---")
display(left.merge(right, on='day'))

print("--- suffixes 로 알아보기 쉽게 ---")
display(left.merge(right, on='day', suffixes=('_주문', '_요일정보')))

In [ ]:
# 병합 덕분에 '주말 여부'로 그룹 요약이 가능해졌습니다.
# 주말(True) 평균 팁이 평일(False)보다 높네요.
merged.groupby('is_weekend')['tip'].mean()

### ⚠️ 행이 늘어나는 함정 — merge 사고 1순위

**merge 는 오른쪽 표에서 키가 맞는 행을 '전부' 붙입니다.** 그래서 오른쪽에 같은 키가 **2번** 있으면 왼쪽의 그 행은 **2줄로 복제**됩니다. 3번이면 3줄이 되죠.

<img src="images/merge_행증식_함정.png" alt="merge 행 증식 함정 — 참조표 키가 중복되면 왼쪽 행이 복제된다" width="900"/>

- **위(정상)**: 참조표에 `Sun`·`Sat` 이 **한 번씩** → 주문 2건이 결과 2행. 행 수 그대로입니다.
- **아래(함정)**: 참조표에 연도가 섞여 `Sun` 이 **2줄** → 주문 1건이 **2줄로 복제**돼 결과가 4행이 됩니다.

**왜 위험할까요?** 에러가 나지 않습니다. 그 상태로 건수를 세거나 금액을 합하면 **조용히 2배**가 되어, 틀린 숫자가 그대로 리포트에 실립니다.

**어떻게 막을까요?**
- 붙이기 전에 `참조표['키'].duplicated().sum()` 이 **0인지 확인**합니다.
- `validate='m:1'` 을 주면 "왼쪽 여러 행 : 오른쪽 한 행" 관계가 아닐 때 **에러로 막아 줍니다**.

In [ ]:
# 참조표에 같은 키가 2번 들어간 상황을 일부러 만들어, 행이 정말 늘어나는지 확인합니다.
# (실무에서는 참조표에 '연도'·'지역' 같은 열이 섞여 들어와 이렇게 됩니다.)
dup_info = pd.concat([day_info.assign(연도=2024), day_info.assign(연도=2025)], ignore_index=True)
print('참조표 행 수:', len(dup_info), '/ 중복된 키 개수:', dup_info['day'].duplicated().sum())

print('정상 참조표로 병합:', df.merge(day_info, on='day').shape)
print('중복 참조표로 병합:', df.merge(dup_info, on='day').shape, '<- 244행이 488행으로!')

# 막는 방법 — validate='m:1' : '왼쪽 여러 행 : 오른쪽 한 행' 이 아니면 에러로 알려 준다
try:
    df.merge(dup_info, on='day', validate='m:1')
except Exception as e:
    print('\nvalidate=\'m:1\' 이 막아 줌 ->', e)

### 🖐️ 함께 따라하기 — 프로그램 정보 붙여 카테고리별 요약
`program_info.csv` 에는 프로그램마다 **카테고리·난이도·정원**이 들어 있습니다. 이 표를 붙이면 원래 없던 기준(카테고리)으로 요약할 수 있어요.

이어서 **참조표에 정보가 빠진 경우**도 겪어 봅니다. 실무 참조표는 늘 한두 개가 비어 있고, 그때 `how=` 를 무엇으로 두느냐가 **행이 사라지느냐 남느냐**를 가릅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) pd.read_csv 로 data/program_info.csv 를 읽어 변수 program_info 에 담고 내용을 확인한다
# 2) gym 과 program_info 를 '프로그램' 키로 merge 해 변수 gym_full 에 담는다
# 3) 병합 뒤 열이 늘었는지 shape 로 확인한다
# 4) gym_full 을 '카테고리' 로 그룹화해 '소모칼로리' 평균을 구해 출력한다
#
# 이제 참조표에 정보가 빠진 상황을 만들어 how= 차이를 확인합니다.
# 5) program_info 에서 '스피닝' 행을 뺀 참조표를 partial 에 담는다
#    (힌트: program_info[program_info['프로그램'] != '스피닝'])
# 6) gym 과 partial 을 기본 방식(inner)으로 병합해 행 수를 출력한다 — 몇 건이 사라졌나?
# 7) 같은 병합을 how='left' 로 다시 해 행 수를 출력한다 — 이번엔 전부 남았나?
# 8) how='left' 결과에서 '카테고리' 열의 결측 개수를 세어 본다 (isna().sum())
#    -> inner 는 짝 없는 행을 '버리고', left 는 '남기고 빈칸(NaN)으로 둔다'

### ✅ 바로 확인 퀴즈
**1.** `how='inner'` 와 `how='left'` 의 차이는?

<details><summary>정답 보기</summary>

inner 는 **양쪽에 다 있는 키**만 남깁니다(교집합). left 는 **왼쪽 표의 키를 전부** 남기고 오른쪽에 짝이 없으면 NaN 을 채웁니다.

</details>

**2.** `df.merge(day_info, on='day')` 의 결과 행 수가 244로 그대로인 이유는?

<details><summary>정답 보기</summary>

day_info 에 4개 요일이 **모두** 있어서 팁 데이터의 모든 행이 짝을 찾기 때문입니다. 빠지는 행이 없어 244행이 유지됩니다.

</details>

---
# 7. concat — 표를 이어 붙이기

## 왜 필요할까요?
merge 가 **키를 맞춰 옆으로** 합치는 것이라면, `concat` 은 **위·아래(또는 옆)로 그냥 이어 붙이는** 것입니다. "1월 데이터 + 2월 데이터"처럼 **같은 구조의 표 여러 개를 쌓을** 때 씁니다.

- `pd.concat([df1, df2])` — 기본은 **행 방향**(세로로 쌓기).
- `ignore_index=True` — 이어 붙인 뒤 인덱스를 0부터 새로 매김(원래 인덱스가 중복되지 않게).
- `axis=1` — **열 방향**(옆으로 붙이기).
- merge 와 차이: merge 는 **공통 키**로 맞춰 합치고, concat 은 **키 없이 위치·이름 그대로** 이어 붙입니다.

### 컬럼이 같아야 하나요?
**원칙은 "같은 구조의 표끼리"** 입니다. 그런데 pandas 는 **컬럼이 달라도 막지 않습니다** — 없는 칸을 `NaN` 으로 채우고 **열을 합집합으로 늘려** 버립니다. 에러가 없으니 알아채기 어렵죠.

- `tip` 과 `Tip` 처럼 **대소문자 하나만 달라도** 열이 두 개로 갈라지고 절반이 `NaN` 이 됩니다.
- `join='inner'` 를 주면 **양쪽에 다 있는 열만** 남깁니다.
- 그래서 이어 붙이기 전에 `df1.columns.equals(df2.columns)` 로 확인하는 습관이 좋습니다.

In [ ]:
# 표의 앞 50행과 뒤 50행을 세로로 이어 붙이기
head_part = df.head(50)
tail_part = df.tail(50)
stacked = pd.concat([head_part, tail_part], ignore_index=True)
print("50 + 50 =", len(stacked), "행")   # 100
stacked.shape

`axis=1` 을 주면 **열 방향(옆으로)** 이어 붙입니다 — 행 위치(인덱스)를 그대로 맞대어 열을 늘리는 것이라, `merge` 처럼 **공통 키로 짝을 찾지 않습니다**. 키를 맞춰 합치려면 merge, 위치 그대로 나란히 붙이려면 `concat(axis=1)` 을 씁니다.

In [ ]:
# axis=1 — 열 방향으로 옆에 이어 붙이기 (같은 행 위치끼리 맞댄다)
left_cols = df[['total_bill', 'tip']].head(3)    # 앞 3행의 금액·팁 두 열
right_cols = df[['day', 'time']].head(3)         # 같은 앞 3행의 요일·시간대 두 열
side_by_side = pd.concat([left_cols, right_cols], axis=1)   # 옆으로 붙여 4열이 됨
print("열 방향 결합 크기:", side_by_side.shape)   # (3, 4)
side_by_side

In [ ]:
# 컬럼 이름이 하나만 달라도? — 에러 없이 열이 갈라지고 NaN 이 생긴다
a = df[['day', 'tip']].head(2)
b = df[['day', 'tip']].tail(2).rename(columns={'tip': 'Tip'})   # 대문자 T 하나 차이
print('열 이름이 같은가?', a.columns.equals(b.columns))

display(pd.concat([a, b]))                  # tip / Tip 두 열로 갈라지고 절반이 NaN
display(pd.concat([a, b], join='inner'))    # 공통 열(day)만 남긴다

### 🖐️ 함께 따라하기 — 두 조각 이어 붙이기
표의 앞뒤 조각을 잘라 **세로로** 이어 붙여 봅시다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) gym 의 앞 30행(head)과 뒤 20행(tail)을 각각 변수에 담는다
# 2) pd.concat 으로 두 조각을 세로로 이어 붙인다 (인덱스는 0부터 다시 매기기)
# 3) 이어 붙인 결과의 행 수가 50인지 len 으로 확인한다

### ✅ 바로 확인 퀴즈
**1.** merge 와 concat 의 차이를 한 문장으로?

<details><summary>정답 보기</summary>

merge 는 **공통 키를 맞춰 옆으로** 합치고, concat 은 **키 없이 위·아래(또는 옆)로 그냥 이어 붙입니다.**

</details>

**2.** `pd.concat` 에서 `axis=0` 과 `axis=1` 은 결과 표의 모양이 어떻게 달라질까요?

<details><summary>정답 보기</summary>

`axis=0`(기본)은 **행 방향(세로)** 으로 쌓아 **행 수가 늘어나고**, `axis=1`은 **열 방향(가로)** 으로 붙여 **열 수가 늘어납니다**. 세로로 쌓을 땐 열 이름을, 가로로 붙일 땐 행 인덱스를 기준으로 정렬합니다.

</details>

---
## 🚀 응용 클론코딩 — 집계로 짧은 리포트 만들기

오늘 배운 것을 **한 흐름**으로 이어 봅시다: 병합 → 여러 키 그룹 → **사용자 정의 집계 + 범주형 집계** → 격자표 → 정렬.

**미션**: 피트니스 데이터에 프로그램 정보를 붙여, **카테고리 × 회원등급별 이용 요약표**를 만듭니다. 평균만 담지 말고 **퍼짐(범위)** 과 **가장 흔한 이용 시간대**까지 한 표에 넣어, "가장 오래 운동하는 조합"과 "그 조합이 주로 언제 오는지"를 함께 찾아냅니다.

> 이렇게 만든 요약표 한 장이 곧 리포트의 근거가 됩니다. 다음 시간에는 이 표를 **그림**으로 바꿉니다.

In [ ]:
# 🖐️ 함께 따라하기 — 집계 리포트 (아래 순서대로 직접 작성해 보세요)
# 1) gym 과 program_info 를 '프로그램' 키로 merge 해 gym_full 에 담는다
# 2) gym_full 을 ['카테고리', '회원등급'] 두 키로 그룹화하고,
#    3절의 '이름 붙이기' 형태로 다섯 열을 한 번에 만든다 — agg(새이름=('열', 집계방법))
#    - 평균운동시간 : '운동시간' 평균
#    - 운동시간범위 : '운동시간' 최댓값 - 최솟값        (lambda)
#    - 평균칼로리   : '소모칼로리' 평균
#    - 주요시간대   : '시간대' 의 최빈값                (범주형 집계, mode()[0])
#    - 방문건수     : 'size'
# 3) 결과를 소수 1자리로 반올림해 출력한다
# 4) pivot_table 로 행='카테고리', 열='회원등급', 값='운동시간' 평균 격자표를 만들어 출력한다
# 5) 2)의 결과를 '평균운동시간' 내림차순으로 정렬해 가장 오래 운동하는 조합을 확인한다
# 6) '주요시간대' 열을 훑어 다른 조합과 다른 시간대를 쓰는 조합이 있는지 찾아본다

---
## 오늘의 인사이트 — 요약이 말해 준 것

집계 몇 줄로 이 식당에 대해 이런 사실들을 알아냈습니다.
- **주말(Sat·Sun)의 평균 팁이 평일보다 높다** — 주말 3.115달러 vs 평일 2.763달러(§6 병합 결과). 식사 금액도 같은 방향인지는 `merged.groupby('is_weekend')['total_bill'].mean()` 으로 직접 확인해 보세요.
- **팁이 가장 높은 요일은 일요일(Sun, 3.255달러)**, 가장 낮은 요일은 금요일(Fri)입니다.
- **인원수가 많을수록 식사 금액이 커진다** — size 1명 7.2달러 → 4명 28.6달러.
- **요일마다 영업 패턴이 다르다** — Thur 는 대부분 점심, Sat·Sun 은 전부 저녁 장사(crosstab).

숫자를 그룹으로 묶어 요약하니, 흩어진 244장의 영수증에서 **패턴**이 드러났습니다. 이것이 EDA(탐색적 데이터 분석)의 첫걸음입니다.

## 이번 강의 정리

| 하고 싶은 일 | 함수 | 쓰임 |
|---|---|---|
| 그룹별 요약 | `df.groupby('키')['열'].mean()` | ~별 평균·합계·개수 |
| 여러 키·여러 집계 | `groupby([...]).agg([...])` | 조합별·여러 통계 한 번에 |
| 내가 만든 집계 | `.agg(이름=('열', lambda s: ...))` | 목록에 없는 계산(범위·비율 등) |
| 여러 열을 함께 | `.apply(내함수)` | 그룹 표 전체로 계산(가중평균 등) |
| 범주형 열 요약 | `.agg(lambda s: s.mode()[0])` · `nunique` | 가장 흔한 값·종류 수(평균은 에러) |
| 행·열 교차 요약표 | `df.pivot_table(index, columns, values, aggfunc)` | 격자 요약(엑셀 피벗) |
| 빈도 교차표 | `pd.crosstab(행, 열)` | 조합별 건수 세기 |
| 두 표를 키로 합치기 | `df.merge(other, on='키', how=...)` | 흩어진 정보 옆으로 결합 |
| 표를 이어 붙이기 | `pd.concat([df1, df2])` | 같은 구조 표 위·아래로 쌓기 |

## ⏭️ 예고 — 다음: 이 요약을 그림으로 (시각화)
오늘은 숫자 표로 요약했습니다. 하지만 "주말이 팁이 높다"는 사실은 **막대그래프 하나**로 훨씬 강렬하게 보여줄 수 있죠.

다음 시간에는 **matplotlib·seaborn** 으로 오늘의 요약을 **그림**으로 바꿉니다 — 분포를 보는 히스토그램·박스플롯, 그룹을 비교하는 막대그래프, 두 값의 관계를 보는 산점도까지. "표로 요약 → 그림으로 전달"이 EDA 의 완성입니다.